# Module 318 — Skills Assessment: Feature Obfuscation Attack

This assessment has two distinct phases. **Phase 1 is white-box:** use the downloaded Naive Bayes model to flip 10 positive movie reviews to negative with at most 30 appended words per review. **Phase 2 is black-box:** use only the `/predict` API to flip 10 negative reviews to positive with at most 40 appended words per review. Both phases must pass.

Start the Academy assessment instance, then set `TARGET_IP` and `TARGET_PORT` below or define the `BASE_URL` environment variable. The notebook does not store a live target address or flag. The downloaded model stays in memory; its pickle is loaded with a restrictive class allowlist.

## The math behind both directions

For Phase 1, score a token by its class log-likelihood difference: `L(w)=log P(w|negative)-log P(w|positive)`. Say it aloud as **“L of w equals log probability of w given negative minus log probability of w given positive.”** A positive value means the learned model associates that feature more strongly with negative reviews. For example, `log(0.02/0.001)=log(20)≈3.00` favors negative. This ranks candidate features; the actual model must verify each growing review because n-grams and existing text also affect its prediction.

For Phase 2, the model is hidden. Measure `Δp_positive(w,x)=p_positive(x + w)-p_positive(x)`. Say it as **“delta positive probability of w on x equals positive probability after adding w minus positive probability before.”** If the score rises from 0.18 to 0.34, the observed increase is 0.16, or 16 percentage points. A positive change is useful, but the returned label is the authoritative flip signal.

In [ ]:
import io
import os
import pickle
import re
import warnings

import numpy as np
import pandas as pd
import requests
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from numpy._core.multiarray import _reconstruct, scalar

# Replace placeholders after starting the target, or set BASE_URL in the
# environment before launching Jupyter.
TARGET_IP = '<TARGET_IP>'
TARGET_PORT = '<PORT>'
BASE_URL = os.getenv(
    'BASE_URL', f'http://{TARGET_IP}:{TARGET_PORT}'
).rstrip('/')
REQUEST_TIMEOUT_SECONDS = 20
MAX_MODEL_BYTES = 20_000_000
TOKEN_PATTERN = re.compile(r'[A-Za-z]+')

if '<TARGET_IP>' in BASE_URL or '<PORT>' in BASE_URL:
    raise ValueError('Set TARGET_IP and TARGET_PORT, or define BASE_URL.')
session = requests.Session()

## API and request helpers

The API helpers check HTTP status and timeouts so connection errors do not masquerade as sentiment results. The challenge metadata supplies review IDs, original texts, target labels, and each phase's word limit. The final submissions preserve those IDs exactly.

In [ ]:
def get_json(path):
    response = session.get(
        f'{BASE_URL}{path}', timeout=REQUEST_TIMEOUT_SECONDS
    )
    response.raise_for_status()
    return response.json()


def post_json(path, payload):
    response = session.post(
        f'{BASE_URL}{path}', json=payload, timeout=REQUEST_TIMEOUT_SECONDS
    )
    response.raise_for_status()
    return response.json()


def added_word_count(original, augmented):
    """Require the exact original prefix and count appended single tokens."""
    if not augmented.startswith(original):
        raise ValueError('Original review must remain an exact prefix')
    suffix = augmented[len(original):]
    if not suffix:
        return 0
    if re.fullmatch(r'(?: [A-Za-z]+)+', suffix) is None:
        raise ValueError('Add only single letter-only words separated by spaces')
    return len(TOKEN_PATTERN.findall(suffix))


health = get_json('/health')
print('Assessment service:', health)
if health.get('status') != 'healthy':
    raise RuntimeError(f'Assessment target is not healthy: {health}')

## Phase 1 — Download and inspect the white-box model

A pickle can execute code when unrestricted globals are reconstructed. Instead of calling plain `pickle.load`, this cell allows only the exact classes found in the HTB bundle: `MultinomialNB`, `CountVectorizer`, and NumPy array/dtype reconstruction types. Any unexpected global raises an error. The response is size-limited and loaded from memory rather than saved as a model file.

This is a constrained loader, not a general safe-pickle format. It is intentionally narrow for this known course bundle; if the bundle changes and requests a new global, stop and inspect the pickle opcodes before extending the allowlist.

In [ ]:
class RestrictedModelUnpickler(pickle.Unpickler):
    """Reconstruct only known NumPy/scikit-learn model bundle components."""
    ALLOWED = {
        ('sklearn.naive_bayes', 'MultinomialNB'): MultinomialNB,
        ('sklearn.feature_extraction.text', 'CountVectorizer'): CountVectorizer,
        ('numpy._core.multiarray', '_reconstruct'): _reconstruct,
        ('numpy._core.multiarray', 'scalar'): scalar,
        ('numpy', 'ndarray'): np.ndarray,
        ('numpy', 'dtype'): np.dtype,
        ('numpy', 'int64'): np.int64,
    }

    def find_class(self, module, name):
        try:
            return self.ALLOWED[(module, name)]
        except KeyError as exc:
            raise pickle.UnpicklingError(
                f'Blocked unexpected pickle global: {module}.{name}'
            ) from exc


model_response = session.get(
    f'{BASE_URL}/model/download', timeout=REQUEST_TIMEOUT_SECONDS
)
model_response.raise_for_status()
model_bytes = model_response.content
if len(model_bytes) > MAX_MODEL_BYTES:
    raise ValueError(f'Model bundle exceeds {MAX_MODEL_BYTES:,} bytes')

bundle = RestrictedModelUnpickler(io.BytesIO(model_bytes)).load()
required_keys = {'classifier', 'vectorizer', 'feature_names'}
if not required_keys.issubset(bundle):
    raise ValueError(f'Model bundle is missing: {required_keys - set(bundle)}')
classifier = bundle['classifier']
vectorizer = bundle['vectorizer']
feature_names = np.asarray(bundle['feature_names']).astype(str)
class_names = [str(label) for label in classifier.classes_]
negative_index = class_names.index('negative')
positive_index = class_names.index('positive')
if len(feature_names) != len(classifier.feature_log_prob_[0]):
    raise ValueError('Feature names do not align with classifier probabilities')

print('Bundle keys:', sorted(bundle))
print('Classifier classes:', class_names)
print('Feature count:', len(feature_names))
print('Model bytes loaded in memory:', len(model_bytes))
del model_bytes

## Rank single-token negative evidence

`feature_log_prob_[class, feature]` stores a logarithm of the learned conditional feature probability. Subtracting positive from negative log probability is equivalent to the log of a probability ratio. We retain alphabetic single-token features because the assessment adds words, not arbitrary vector columns. The vectorizer also learns n-grams; the ranking is only an initial guide, and local re-prediction checks the full feature effects.

In [ ]:
negative_log_probability = classifier.feature_log_prob_[negative_index]
positive_log_probability = classifier.feature_log_prob_[positive_index]
negative_log_advantage = negative_log_probability - positive_log_probability

whitebox_feature_records = []
for feature, score in zip(feature_names, negative_log_advantage):
    if re.fullmatch(r'[A-Za-z]+', feature):
        whitebox_feature_records.append({
            'word': feature.lower(),
            'negative_log_advantage': float(score),
        })
whitebox_word_table = pd.DataFrame(whitebox_feature_records).drop_duplicates(
    'word'
).sort_values('negative_log_advantage', ascending=False, ignore_index=True)
display(whitebox_word_table.head(20))


def local_sentiment(text):
    """Verify a review with the downloaded model and class-index mapping."""
    probabilities = classifier.predict_proba(vectorizer.transform([text]))[0]
    label = str(classifier.classes_[int(np.argmax(probabilities))])
    return {
        'label': label,
        'negative_probability': float(probabilities[negative_index]),
        'positive_probability': float(probabilities[positive_index]),
    }

## Build and verify Phase 1 solutions

For each positive review, try the highest negative-log-advantage words that do not already appear in the text. After each append, ask the downloaded model to classify the entire review. Stop as soon as it predicts negative or the per-review cap is reached. The output retains the exact review ID and a local prediction for review before submission.

In [ ]:
whitebox_challenge = get_json('/challenge/whitebox')
whitebox_reviews = whitebox_challenge['reviews']
whitebox_word_limit = int(whitebox_challenge['max_added_words'])
if len(whitebox_reviews) != 10:
    raise ValueError(f'Expected 10 white-box reviews, got {len(whitebox_reviews)}')
if whitebox_word_limit > 30:
    raise ValueError(f'Unexpected white-box word limit: {whitebox_word_limit}')

ranked_negative_words = whitebox_word_table['word'].tolist()
whitebox_solutions = []
whitebox_audit = []
for review in whitebox_reviews:
    original = review['text']
    target = str(review['target_sentiment']).lower()
    if target != 'negative':
        raise ValueError(f'Unexpected target for {review["id"]}: {target}')

    original_word_set = set(TOKEN_PATTERN.findall(original.lower()))
    additions = []
    augmented = original
    prediction = local_sentiment(augmented)
    for word in ranked_negative_words:
        if prediction['label'] == target or len(additions) >= whitebox_word_limit:
            break
        if word in original_word_set or word in additions:
            continue
        proposed = augmented + ' ' + word
        if added_word_count(original, proposed) > whitebox_word_limit:
            break
        augmented = proposed
        additions.append(word)
        prediction = local_sentiment(augmented)

    count = added_word_count(original, augmented)
    if prediction['label'] != target:
        raise RuntimeError(
            f'{review["id"]} did not flip within {whitebox_word_limit} words; '
            'inspect whitebox_audit and adjust the ranking strategy.'
        )
    whitebox_solutions.append({
        'id': review['id'],
        'augmented_text': augmented,
    })
    whitebox_audit.append({
        'id': review['id'],
        'words_added': count,
        'added_words': additions,
        **prediction,
    })

whitebox_audit_df = pd.DataFrame(whitebox_audit)
display(whitebox_audit_df)
assert len(whitebox_solutions) == 10
assert all(row['label'] == 'negative' for row in whitebox_audit)

## Submit Phase 1

Before submission, the notebook checks all 10 IDs, confirms every local model prediction is negative, and enforces each 30-word limit. The server's response is authoritative; keep its per-review feedback available in case a model-version difference causes a rejection.

In [ ]:
whitebox_feedback = post_json(
    '/submit/whitebox', {'solutions': whitebox_solutions}
)
print(whitebox_feedback)

## Phase 2 — Query-only positive sentiment flips

Do not use `bundle`, `classifier`, `vectorizer`, or the white-box feature ranking below. The black-box candidate vocabulary is a separate, human-curated list. For every negative review, query its baseline and the same candidate words with `/predict`, rank by the increase in positive probability, then test growing prefixes until the API label is positive. This is an output-only attack.

In [ ]:
blackbox_prediction_queries = 0
blackbox_query_log = []

def api_predict(text):
    """Use only the documented score API; do not consult local model state."""
    global blackbox_prediction_queries
    result = post_json('/predict', {'text': text})
    required = {'label', 'positive_probability', 'negative_probability'}
    if not required.issubset(result):
        raise ValueError(f'Unexpected /predict fields: {list(result)}')
    result = {
        'label': str(result['label']).lower(),
        'positive_probability': float(result['positive_probability']),
        'negative_probability': float(result['negative_probability']),
    }
    if not all(0.0 <= result[k] <= 1.0 for k in
               ('positive_probability', 'negative_probability')):
        raise ValueError(f'Invalid prediction probabilities: {result}')
    blackbox_prediction_queries += 1
    blackbox_query_log.append({
        'query_number': blackbox_prediction_queries,
        'text_characters': len(text),
        **result,
    })
    return result


positive_candidate_words = [
    'excellent', 'wonderful', 'engaging', 'captivating', 'masterpiece',
    'brilliant', 'outstanding', 'delightful', 'enjoyable', 'entertaining',
    'charming', 'superb', 'moving', 'compelling', 'heartfelt', 'recommend',
    'impressive', 'memorable', 'funny', 'hilarious', 'beautiful', 'fantastic',
    'best', 'love', 'great', 'perfect', 'fresh', 'fascinating', 'inspiring',
    'touching', 'strong', 'smart', 'favorite', 'success', 'clever', 'sweet',
    'powerful', 'terrific', 'beloved', 'admire', 'joyful', 'pleasant', 'likable',
    'enthralling', 'gripping', 'exciting', 'riveting', 'wonderfully',
    'choreography', 'cinematic', 'authentic', 'satisfying', 'enjoyed', 'liked',
    'loved', 'amazing', 'brilliantly', 'performances', 'actors', 'storytelling',
    'original', 'creative', 'unique', 'adventure', 'fun', 'entertainment',
    'emotion', 'emotional', 'warm', 'uplifting', 'charm', 'talented',
    'masterful', 'perfectly', 'recommended', 'worthwhile', 'classic', 'genius',
    'romantic', 'beautifully', 'fantasy', 'delight', 'pleasure', 'successfully',
    'engrossing', 'suspenseful', 'funniest', 'delightfully', 'effective', 'well',
    'cast', 'direction', 'screenplay', 'story', 'performance', 'character',
    'cinema', 'visual', 'soundtrack', 'harmony', 'vibrant', 'engrossing',
]
BLACKBOX_PROBE_WORDS = len(positive_candidate_words)
if any(re.fullmatch(r'[A-Za-z]+', w) is None for w in positive_candidate_words):
    raise ValueError('Black-box candidates must be single letter-only words')

Each API query returns both class probabilities and a label. The difference `p_positive(augmented)-p_positive(original)` estimates a word's direction on that specific review. A word's single-token effect can change after other words are appended, so the solver verifies the current prefix after every addition and stops on the API's positive label.

In [ ]:
blackbox_challenge = get_json('/challenge/blackbox')
blackbox_reviews = blackbox_challenge['reviews']
blackbox_word_limit = int(blackbox_challenge['max_added_words'])
if len(blackbox_reviews) != 10:
    raise ValueError(f'Expected 10 black-box reviews, got {len(blackbox_reviews)}')
if blackbox_word_limit > 40:
    raise ValueError(f'Unexpected black-box word limit: {blackbox_word_limit}')

blackbox_solutions = []
blackbox_audit = []
blackbox_failures = []
for review in blackbox_reviews:
    original = review['text']
    target = str(review['target_sentiment']).lower()
    if target != 'positive':
        raise ValueError(f'Unexpected target for {review["id"]}: {target}')

    baseline = api_predict(original)
    original_word_set = set(TOKEN_PATTERN.findall(original.lower()))
    candidate_impacts = []
    for word in positive_candidate_words[:BLACKBOX_PROBE_WORDS]:
        if word in original_word_set:
            continue
        result = api_predict(original + ' ' + word)
        candidate_impacts.append({
            'word': word,
            'positive_probability': result['positive_probability'],
            'impact': result['positive_probability'] - baseline['positive_probability'],
        })

    ranked = sorted(candidate_impacts, key=lambda row: row['impact'], reverse=True)
    useful = [row['word'] for row in ranked if row['impact'] > 0]
    if not useful:
        useful = [row['word'] for row in ranked]

    augmented = original
    additions = []
    prediction = baseline
    for word in useful:
        if prediction['label'] == target or len(additions) >= blackbox_word_limit:
            break
        proposed = augmented + ' ' + word
        if added_word_count(original, proposed) > blackbox_word_limit:
            break
        prediction = api_predict(proposed)
        augmented = proposed
        additions.append(word)

    count = added_word_count(original, augmented)
    audit_row = {
        'id': review['id'],
        'words_added': count,
        'added_words': additions,
        **prediction,
    }
    blackbox_audit.append(audit_row)
    if prediction['label'] == target:
        blackbox_solutions.append({
            'id': review['id'],
            'augmented_text': augmented,
        })
    else:
        blackbox_failures.append(review['id'])

blackbox_audit_df = pd.DataFrame(blackbox_audit)
display(blackbox_audit_df)
print('Black-box prediction requests:', blackbox_prediction_queries)
if blackbox_failures:
    print('Reviews that need another candidate/sequence pass:', blackbox_failures)
    raise RuntimeError('Not submitting until all 10 reviews predict positive.')
assert len(blackbox_solutions) == 10
assert all(row['label'] == 'positive' for row in blackbox_audit)

## Submit Phase 2 and confirm assessment status

The notebook submits only after all 10 API labels are positive and each appended suffix is under 40 words. Check the server's response and `/status` for completion. The final response may contain a flag; this notebook has no saved outputs, and the reusable notes deliberately omit it.

In [ ]:
blackbox_feedback = post_json(
    '/submit/blackbox', {'solutions': blackbox_solutions}
)
print(blackbox_feedback)

assessment_status = get_json('/status')
print('Assessment status:', assessment_status)

## COAE quick reference

- Phase 1 is white-box negative-target evidence: `log P(w|negative)-log P(w|positive)`. Verify each growing text with the downloaded model.
- Phase 2 is black-box positive-target score change: `p_positive(augmented)-p_positive(original)`. Use only `/predict` outputs for ranking.
- Keep the modes separate: no Phase 1 model weights or feature ranking in Phase 2.
- Append no more than 30 words per white-box review and 40 per black-box review. Submit all 10 correct IDs for each phase.
- The server's submission feedback and `/status` determine success; local predictions are not a substitute for server validation.
- Pickle files can run code during loading. This notebook's restricted loader allows only the specific model classes needed for this HTB bundle.